In [2]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import ZenodoSource
from imgtools.nifti.crawl import Crawler
from rich import print
import pandas as pd

/home/gpudual/bhklab/declan/med-image_index/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# COVID-19 CT

Completed

In [ ]:
indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])

21:03:56 [info     ] Indexed datasets path: /home/gpudual/bhklab/declan/med-image_index/indexed_datasets [imgnet] call=store.__init__:71


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'AMOS',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung',
    'B-mode-and-CEUS-Liver'
]

In [4]:
description = """
This dataset contains 20 labeled COVID-19 CT scans. Left lung, right lung, and infections are labeled by two radiologists and verified by an experienced radiologist. 
To promote the studies of annotation-efficient deep learning methods, we set up three segmentation benchmark tasks based on this dataset https://gitee.com/junma11/COVID-19-CT-Seg-Benchmark.

In particular, we focus on learning to segment left lung, right lung, and infections using

    pure but limited COVID-19 CT scans;
    existing labeled lung CT dataset from other non-COVID-19 lung diseases;
    heterogeneous datasets include both COVID-19 and non-COVID-19 CT scans
"""

In [5]:
source = ZenodoSource(
    file_type="nifti",
    source="zenodo",
    record_id="3757476",
    filenames=["COVID-19-CT-Seg_20cases.zip", "Infection_Mask.zip", "Lung_and_Infection_Mask.zip", "Lung_Mask.zip"],
    description=description,
    #post_download=["unzip"]
)

In [11]:
dataset_index_path = (indexed_datasets.path / "COVID-19 CT Lung")
dataset_index_path.mkdir(parents=True, exist_ok=True)
(dataset_index_path / "parquet").mkdir(exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [6]:

print(indexed_datasets.source_config("COVID-19 CT Lung"))

ZenodoSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    source='zenodo',
    record_id='3757476',
    filenames=[
        'COVID-19-CT-Seg_20cases.zip',
        'Infection_Mask.zip',
        'Lung_and_Infection_Mask.zip',
        'Lung_Mask.zip'
    ],
    post_download=['unzip'],
    description='\nThis dataset contains 20 labeled COVID-19 CT scans. Left lung, right lung, and infections are 
labeled by two radiologists and verified by an experienced radiologist. \nTo promote the studies of 
annotation-efficient deep learning methods, we set up three segmentation benchmark tasks based on this dataset 
https://gitee.com/junma11/COVID-19-CT-Seg-Benchmark.\n\nIn particular, we focus on learning to segment left lung, 
right lung, and infections using\n\n    pure but limited COVID-19 CT scans;\n    existing labeled lung CT dataset 
from other non-COVID-19 lung diseases;\n    heterogeneous datasets include both COVID-19 and non-COVID-19 CT 
scans\n'
)

In [7]:
temp_data_path = Path("temp_data") / "COVID-19 CT Lung"
temp_data_path.mkdir(parents=True, exist_ok=True)
#downloader = indexed_datasets.downloader("COVID-19 CT Lung").download(temp_data_path)

In [8]:
import zipfile
from glob import glob
import os

def extract_zip(zip_path, extract_to=None):
    """
    Extract a zip file to a specified directory
    
    Args:
        zip_path: Path to the zip file
        extract_to: Directory to extract to (default: same directory as zip)
    """
    if extract_to is None:
        extract_to = os.path.dirname(zip_path)
    
    # Create extraction directory if it doesn't exist
    os.makedirs(extract_to, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
        print(f"Extracted {zip_path} to {extract_to}")

for zip_file in glob("temp_data/COVID-19 CT Lung/*.zip"):
    print(zip_file)
    # Get the base name without extension to create a folder with the same name
    zip_name = os.path.splitext(os.path.basename(zip_file))[0]
    extract_to = os.path.join("temp_data/COVID-19 CT Lung", zip_name)
    extract_zip(zip_file, extract_to)

temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases.zip

BadZipFile: File is not a zip file

In [9]:
crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="COVID-19-CT-Seg_20cases/{_throwaway}_{ImageID}.nii.gz",
    mask_name_pattern="{ROINames}_Mask/{_throwaway}_{ImageID}.nii.gz",
    output_dir=indexed_datasets.path,
    n_jobs=4,
    force=True,
    deep=True,
)
crawler.crawl()


Parsing 80 files:   0%|                                                                                                                                                                                                           | 0/80 [00:00<?, ?it/s]

Parsing 80 files:  52%|█████████████████████████████████████████████████████████████████████████████████████████████████████▊                                                                                            | 42/80 [09:24<12:21, 19.51s/it]

processing /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_004.nii.gz...
read /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_004.nii.gz
processing scan /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_004.nii.gz
processed /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_004.nii.gz
updated /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_004.nii.gz
processing /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_007.nii.gz...
read /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_007.nii.gz
processing scan /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg

Parsing 80 files:  56%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                                                                                    | 45/80 [16:10<26:16, 45.05s/it]

processing /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_003.nii.gz...
read /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_003.nii.gz
processing scan /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_003.nii.gz
processed /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_003.nii.gz
updated /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_003.nii.gz
processing /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_006.nii.gz...
read /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_006.nii.gz
processing scan /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg

Parsing 80 files:  61%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                                                                           | 49/80 [18:42<22:45, 44.03s/it]

processing /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_002.nii.gz...
read /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_002.nii.gz
processing scan /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_002.nii.gz
processed /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_002.nii.gz
updated /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_002.nii.gz
processing /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_005.nii.gz...
read /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases/coronacases_005.nii.gz
processing scan /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/COVID-19-CT-Seg

Parsing 80 files:  65%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                                                                  | 52/80 [42:57<1:31:09, 195.35s/it]


processing /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/Lung_Mask/radiopaedia_10_85902_1.nii.gz...
read /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/Lung_Mask/radiopaedia_10_85902_1.nii.gz
processing mask /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/Lung_Mask/radiopaedia_10_85902_1.nii.gz
processed /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/Lung_Mask/radiopaedia_10_85902_1.nii.gz
updated /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/Lung_Mask/radiopaedia_10_85902_1.nii.gz
processing /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/Lung_Mask/radiopaedia_14_85914_0.nii.gz...
read /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/Lung_Mask/radiopaedia_14_85914_0.nii.gz
processing mask /home/gpudual/bhklab/declan/med-image_index/temp_data/COVID-19 CT Lung/Lung_Mask/radiopaedia_14_85914_0.nii.gz
processed /home/gpudual

In [10]:
index_df = pd.read_csv(indexed_datasets.path / "COVID-19 CT Lung/index.csv")

index_df



,_throwaway,ImageID,ROINames,filepath,file_type,reference_id,class,hash,size,ndim,...,mask.bbox.max_coord,mask.feret_diameter,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan
0,coronacases,1,NaN,COVID-19-CT-Seg_20cases/coronacases_001.nii.gz,scan,coronacases_001,MedImage,01044c003507f6d47a7d30582454f37647d808f4,"(512, 512, 301)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,coronacases,2,NaN,COVID-19-CT-Seg_20cases/coronacases_002.nii.gz,scan,coronacases_002,MedImage,d00b555fbb74bdfb0d1a24c36a86c77cf1b1f821,"(512, 512, 200)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,coronacases,3,NaN,COVID-19-CT-Seg_20cases/coronacases_003.nii.gz,scan,coronacases_003,MedImage,38521f9fe8f0f84d2d61611fcb82dc332c5ad3b8,"(512, 512, 200)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,coronacases,4,NaN,COVID-19-CT-Seg_20cases/coronacases_004.nii.gz,scan,coronacases_004,MedImage,ec6ebbd03d7341eb2500c1ffa0c464d63463eb54,"(512, 512, 270)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,coronacases,5,NaN,COVID-19-CT-Seg_20cases/coronacases_005.nii.gz,scan,coronacases_005,MedImage,c98d379eadf987210a755de2f382df1f4cb06899,"(512, 512, 290)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,radiopaedia_29_86491,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_29_86491_1...,mask,radiopaedia_29_86491_1,Mask,f9356b1451cb615ac92a1c4c0b5f5221a8b5618d,"(630, 630, 42)",3,...,"(577, 458, 40)",349.214496,0.585030,1.201023,1.791228,111.104281,155121.305102,"(161.93577705223717, 194.48866494429925, 348.3...",2.0,COVID-19-CT-Seg_20cases/radiopaedia_29_86491_1...
76,radiopaedia_36_86526,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_36_86526_0...,mask,radiopaedia_36_86526_0,Mask,ee1d70f77ead45a22dbac5208efa16fc186e2f30,"(630, 630, 45)",3,...,"(515, 445, 45)",344.584211,0.682770,1.521577,1.006372,117.039702,172137.812762,"(176.56902574758502, 268.6633141859198, 270.37...",1.0,COVID-19-CT-Seg_20cases/radiopaedia_36_86526_0...
77,radiopaedia_40_86625,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_40_86625_0...,mask,radiopaedia_40_86625_0,Mask,98bd91bde83079c5388cd87b6aed9ca15a6fac5a,"(630, 630, 93)",3,...,"(537, 512, 85)",339.644040,0.492754,1.639159,1.258538,104.970787,138467.155967,"(139.87142807741324, 229.2714811254359, 288.54...",2.0,COVID-19-CT-Seg_20cases/radiopaedia_40_86625_0...
78,radiopaedia_4_85506,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_4_85506_1....,mask,radiopaedia_4_85506_1,Mask,8014de8723588983711ca63aa32bab880e042e2f,"(630, 630, 39)",3,...,"(622, 471, 36)",408.425614,0.482248,1.501956,1.514034,109.767139,151409.998915,"(145.77586664720556, 218.94898355345663, 331.4...",3.0,COVID-19-CT-Seg_20cases/radiopaedia_4_85506_1....


In [11]:
index_df

,_throwaway,ImageID,ROINames,filepath,file_type,reference_id,class,hash,size,ndim,...,mask.bbox.max_coord,mask.feret_diameter,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan
0,coronacases,1,NaN,COVID-19-CT-Seg_20cases/coronacases_001.nii.gz,scan,coronacases_001,MedImage,01044c003507f6d47a7d30582454f37647d808f4,"(512, 512, 301)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,coronacases,2,NaN,COVID-19-CT-Seg_20cases/coronacases_002.nii.gz,scan,coronacases_002,MedImage,d00b555fbb74bdfb0d1a24c36a86c77cf1b1f821,"(512, 512, 200)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,coronacases,3,NaN,COVID-19-CT-Seg_20cases/coronacases_003.nii.gz,scan,coronacases_003,MedImage,38521f9fe8f0f84d2d61611fcb82dc332c5ad3b8,"(512, 512, 200)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,coronacases,4,NaN,COVID-19-CT-Seg_20cases/coronacases_004.nii.gz,scan,coronacases_004,MedImage,ec6ebbd03d7341eb2500c1ffa0c464d63463eb54,"(512, 512, 270)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,coronacases,5,NaN,COVID-19-CT-Seg_20cases/coronacases_005.nii.gz,scan,coronacases_005,MedImage,c98d379eadf987210a755de2f382df1f4cb06899,"(512, 512, 290)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,radiopaedia_29_86491,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_29_86491_1...,mask,radiopaedia_29_86491_1,Mask,f9356b1451cb615ac92a1c4c0b5f5221a8b5618d,"(630, 630, 42)",3,...,"(577, 458, 40)",349.214496,0.585030,1.201023,1.791228,111.104281,155121.305102,"(161.93577705223717, 194.48866494429925, 348.3...",2.0,COVID-19-CT-Seg_20cases/radiopaedia_29_86491_1...
76,radiopaedia_36_86526,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_36_86526_0...,mask,radiopaedia_36_86526_0,Mask,ee1d70f77ead45a22dbac5208efa16fc186e2f30,"(630, 630, 45)",3,...,"(515, 445, 45)",344.584211,0.682770,1.521577,1.006372,117.039702,172137.812762,"(176.56902574758502, 268.6633141859198, 270.37...",1.0,COVID-19-CT-Seg_20cases/radiopaedia_36_86526_0...
77,radiopaedia_40_86625,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_40_86625_0...,mask,radiopaedia_40_86625_0,Mask,98bd91bde83079c5388cd87b6aed9ca15a6fac5a,"(630, 630, 93)",3,...,"(537, 512, 85)",339.644040,0.492754,1.639159,1.258538,104.970787,138467.155967,"(139.87142807741324, 229.2714811254359, 288.54...",2.0,COVID-19-CT-Seg_20cases/radiopaedia_40_86625_0...
78,radiopaedia_4_85506,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_4_85506_1....,mask,radiopaedia_4_85506_1,Mask,8014de8723588983711ca63aa32bab880e042e2f,"(630, 630, 39)",3,...,"(622, 471, 36)",408.425614,0.482248,1.501956,1.514034,109.767139,151409.998915,"(145.77586664720556, 218.94898355345663, 331.4...",3.0,COVID-19-CT-Seg_20cases/radiopaedia_4_85506_1....


In [12]:
def map_modality(row):
    if row["file_type"] == "scan":
            return "CT"
    else:
        return "SEG"
    

index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df["BodyPartExamined"] = "LUNGS"


index_df

,_throwaway,ImageID,ROINames,filepath,file_type,reference_id,class,hash,size,ndim,...,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,Modality,BodyPartExamined
0,coronacases,1,NaN,COVID-19-CT-Seg_20cases/coronacases_001.nii.gz,scan,coronacases_001,MedImage,01044c003507f6d47a7d30582454f37647d808f4,"(512, 512, 301)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,LUNGS
1,coronacases,2,NaN,COVID-19-CT-Seg_20cases/coronacases_002.nii.gz,scan,coronacases_002,MedImage,d00b555fbb74bdfb0d1a24c36a86c77cf1b1f821,"(512, 512, 200)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,LUNGS
2,coronacases,3,NaN,COVID-19-CT-Seg_20cases/coronacases_003.nii.gz,scan,coronacases_003,MedImage,38521f9fe8f0f84d2d61611fcb82dc332c5ad3b8,"(512, 512, 200)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,LUNGS
3,coronacases,4,NaN,COVID-19-CT-Seg_20cases/coronacases_004.nii.gz,scan,coronacases_004,MedImage,ec6ebbd03d7341eb2500c1ffa0c464d63463eb54,"(512, 512, 270)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,LUNGS
4,coronacases,5,NaN,COVID-19-CT-Seg_20cases/coronacases_005.nii.gz,scan,coronacases_005,MedImage,c98d379eadf987210a755de2f382df1f4cb06899,"(512, 512, 290)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,LUNGS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,radiopaedia_29_86491,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_29_86491_1...,mask,radiopaedia_29_86491_1,Mask,f9356b1451cb615ac92a1c4c0b5f5221a8b5618d,"(630, 630, 42)",3,...,0.585030,1.201023,1.791228,111.104281,155121.305102,"(161.93577705223717, 194.48866494429925, 348.3...",2.0,COVID-19-CT-Seg_20cases/radiopaedia_29_86491_1...,SEG,LUNGS
76,radiopaedia_36_86526,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_36_86526_0...,mask,radiopaedia_36_86526_0,Mask,ee1d70f77ead45a22dbac5208efa16fc186e2f30,"(630, 630, 45)",3,...,0.682770,1.521577,1.006372,117.039702,172137.812762,"(176.56902574758502, 268.6633141859198, 270.37...",1.0,COVID-19-CT-Seg_20cases/radiopaedia_36_86526_0...,SEG,LUNGS
77,radiopaedia_40_86625,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_40_86625_0...,mask,radiopaedia_40_86625_0,Mask,98bd91bde83079c5388cd87b6aed9ca15a6fac5a,"(630, 630, 93)",3,...,0.492754,1.639159,1.258538,104.970787,138467.155967,"(139.87142807741324, 229.2714811254359, 288.54...",2.0,COVID-19-CT-Seg_20cases/radiopaedia_40_86625_0...,SEG,LUNGS
78,radiopaedia_4_85506,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_4_85506_1....,mask,radiopaedia_4_85506_1,Mask,8014de8723588983711ca63aa32bab880e042e2f,"(630, 630, 39)",3,...,0.482248,1.501956,1.514034,109.767139,151409.998915,"(145.77586664720556, 218.94898355345663, 331.4...",3.0,COVID-19-CT-Seg_20cases/radiopaedia_4_85506_1....,SEG,LUNGS


In [13]:
from imgnet.collections.store import IndexedDatasets
from imgindex.model import validate_index
from imgindex.parquet import csv_to_parquet


index_df.rename(columns={"reference_id": "SampleID"}, inplace=True)
index_df.drop(columns=["_throwaway"], inplace=True)
index_df.to_csv(indexed_datasets.path / "COVID-19 CT Lung" / "index.csv", index=False)
csv_to_parquet(index_df, indexed_datasets.path / "COVID-19 CT Lung")
validate_index(index_df, "nifti", lazy=True)

,ImageID,ROINames,filepath,file_type,SampleID,class,hash,size,ndim,nvoxels,...,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,Modality,BodyPartExamined
0,1,NaN,COVID-19-CT-Seg_20cases/coronacases_001.nii.gz,scan,coronacases_001,MedImage,01044c003507f6d47a7d30582454f37647d808f4,"(512, 512, 301)",3,78905344,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,LUNGS
1,2,NaN,COVID-19-CT-Seg_20cases/coronacases_002.nii.gz,scan,coronacases_002,MedImage,d00b555fbb74bdfb0d1a24c36a86c77cf1b1f821,"(512, 512, 200)",3,52428800,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,LUNGS
2,3,NaN,COVID-19-CT-Seg_20cases/coronacases_003.nii.gz,scan,coronacases_003,MedImage,38521f9fe8f0f84d2d61611fcb82dc332c5ad3b8,"(512, 512, 200)",3,52428800,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,LUNGS
3,4,NaN,COVID-19-CT-Seg_20cases/coronacases_004.nii.gz,scan,coronacases_004,MedImage,ec6ebbd03d7341eb2500c1ffa0c464d63463eb54,"(512, 512, 270)",3,70778880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,LUNGS
4,5,NaN,COVID-19-CT-Seg_20cases/coronacases_005.nii.gz,scan,coronacases_005,MedImage,c98d379eadf987210a755de2f382df1f4cb06899,"(512, 512, 290)",3,76021760,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT,LUNGS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_29_86491_1...,mask,radiopaedia_29_86491_1,Mask,f9356b1451cb615ac92a1c4c0b5f5221a8b5618d,"(630, 630, 42)",3,16669800,...,0.585030,1.201023,1.791228,111.104281,155121.305102,"(161.93577705223717, 194.48866494429925, 348.3...",2.0,COVID-19-CT-Seg_20cases/radiopaedia_29_86491_1...,SEG,LUNGS
76,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_36_86526_0...,mask,radiopaedia_36_86526_0,Mask,ee1d70f77ead45a22dbac5208efa16fc186e2f30,"(630, 630, 45)",3,17860500,...,0.682770,1.521577,1.006372,117.039702,172137.812762,"(176.56902574758502, 268.6633141859198, 270.37...",1.0,COVID-19-CT-Seg_20cases/radiopaedia_36_86526_0...,SEG,LUNGS
77,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_40_86625_0...,mask,radiopaedia_40_86625_0,Mask,98bd91bde83079c5388cd87b6aed9ca15a6fac5a,"(630, 630, 93)",3,36911700,...,0.492754,1.639159,1.258538,104.970787,138467.155967,"(139.87142807741324, 229.2714811254359, 288.54...",2.0,COVID-19-CT-Seg_20cases/radiopaedia_40_86625_0...,SEG,LUNGS
78,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_4_85506_1....,mask,radiopaedia_4_85506_1,Mask,8014de8723588983711ca63aa32bab880e042e2f,"(630, 630, 39)",3,15479100,...,0.482248,1.501956,1.514034,109.767139,151409.998915,"(145.77586664720556, 218.94898355345663, 331.4...",3.0,COVID-19-CT-Seg_20cases/radiopaedia_4_85506_1....,SEG,LUNGS
